# 🏥 Real Dataset Multi-Model Training & Benchmark Suite (With Upload Buttons)

This Google Colab notebook lets you **upload your own dataset or individual X-ray images** directly using interactive upload buttons.

### Models Compared:
1. **Baseline ResNet-18** (Classic baseline)
2. **Proposed Attentive-LungNet** (ResNet-18 + CBAM Attention + Dual Head Severity Regressor)
3. **Baseline DenseNet-121** (CheXNet standard)
4. **Baseline EfficientNet-B0** (Google compound scaled)

### Outputs:
- **Multi-Panel Confusion Matrices** on real test images
- **Epoch Learning Curves** (across all epochs requested by your professor)
- **Single-Image Upload Test:** Upload an image to compare all models' diagnosis side-by-side
- **Summary Benchmark Table** (Accuracy, Sensitivity/Recall, Specificity, F1, ROC-AUC)

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q tabulate scikit-learn seaborn matplotlib pandas torchvision torch

import os
import time
import copy
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from google.colab import files

from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Active Hardware: {device}")
if device.type == "cuda":
    print(f"[*] GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: 📤 UPLOAD YOUR DATASET (Interactive Upload Button)

Run this cell and click **'Choose Files'** to upload your dataset `.zip` file (e.g. `chest_xray.zip`). It will automatically extract and configure everything.

In [ ]:
print("👇 Click the button below to upload your dataset zip file (e.g. chest_xray.zip):")
uploaded_files = files.upload()

DATASET_ROOT = "/content/dataset"
os.makedirs(DATASET_ROOT, exist_ok=True)

for filename in uploaded_files.keys():
    if filename.endswith(".zip"):
        print(f"[*] Extracting {filename} to {DATASET_ROOT}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(DATASET_ROOT)
        print("[*] Extraction Complete!")
        break

# If extracted into a nested folder like /content/dataset/chest_xray, adjust path automatically
subdirs = os.listdir(DATASET_ROOT)
if len(subdirs) == 1 and os.path.isdir(os.path.join(DATASET_ROOT, subdirs[0])):
    DATASET_ROOT = os.path.join(DATASET_ROOT, subdirs[0])

print(f"[*] Ready! Dataset root folder: {DATASET_ROOT}")
!ls "{DATASET_ROOT}"

## Step 3: Define Model Architectures & CBAM Attention

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        reduced = max(in_planes // ratio, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, reduced, 1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size=7)
    def forward(self, x):
        out = x * self.ca(x)
        return out * self.sa(out)

# 1. Baseline ResNet-18
class ResNet18Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.net.fc = nn.Linear(self.net.fc.in_features, num_classes)
    def forward(self, x):
        return self.net(x)

# 2. Proposed Attentive-LungNet
class AttentiveLungNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )
        self.cbam = CBAM(in_planes=512)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, num_classes))
        self.severity_head = nn.Sequential(nn.Linear(512, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x, return_severity=False):
        x = self.features(x)
        x = self.cbam(x)
        feat = torch.flatten(self.avgpool(x), 1)
        logits = self.classifier(feat)
        if return_severity:
            return logits, self.severity_head(feat) * 100.0
        return logits

# 3. Baseline DenseNet-121
class DenseNet121Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.net.classifier = nn.Linear(self.net.classifier.in_features, num_classes)
    def forward(self, x):
        return self.net(x)

# 4. Baseline EfficientNet-B0
class EfficientNetB0Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.net.classifier[1] = nn.Linear(self.net.classifier[1].in_features, num_classes)
    def forward(self, x):
        return self.net(x)

## Step 4: Run Multi-Epoch Training & Test Evaluation
Set your desired number of epochs (e.g. `EPOCHS = 15`):

In [ ]:
EPOCHS = 15  # Change to 10, 15, or 20 epochs as requested
BATCH_SIZE = 32
LR = 1e-4

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Loaders
train_ds = datasets.ImageFolder(os.path.join(DATASET_ROOT, "train"), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATASET_ROOT, "val"), transform=eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATASET_ROOT, "test"), transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
class_names  = train_ds.classes
print(f"[*] Datasets Loaded: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

# Models to train
models_dict = {
    "ResNet-18 (Baseline)": ResNet18Baseline().to(device),
    "Attentive-LungNet (Proposed)": AttentiveLungNet().to(device),
    "DenseNet-121 (CheXNet)": DenseNet121Baseline().to(device),
    "EfficientNet-B0 (Baseline)": EfficientNetB0Baseline().to(device)
}

trained_models = {}
histories = {}
test_metrics = {}

for name, model in models_dict.items():
    print(f"\n>>> TRAINING {name} ({EPOCHS} EPOCHS) <<<")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(x_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * x_b.size(0)
            correct += (torch.argmax(out, dim=1) == y_b).sum().item()
            total += y_b.size(0)
        t_acc = (correct / total) * 100.0
        
        # Validation
        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for x_b, y_b in val_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                out = model(x_b)
                v_loss += criterion(out, y_b).item() * x_b.size(0)
                v_corr += (torch.argmax(out, dim=1) == y_b).sum().item()
                v_tot += y_b.size(0)
        val_acc = (v_corr / v_tot) * 100.0
        scheduler.step()
        
        hist["train_loss"].append(r_loss / total)
        hist["val_loss"].append(v_loss / v_tot)
        hist["train_acc"].append(t_acc)
        hist["val_acc"].append(val_acc)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Acc: {t_acc:.1f}% | Val Acc: {val_acc:.1f}%")
        
    model.load_state_dict(best_wts)
    trained_models[name] = model
    histories[name] = hist
    
    # Test evaluation
    model.eval()
    preds, targets, probs = [], [], []
    with torch.no_grad():
        for x_b, y_b in test_loader:
            out = model(x_b.to(device))
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            targets.extend(y_b.numpy())
            
    cm = confusion_matrix(targets, preds)
    test_metrics[name] = {
        "Accuracy (%)": accuracy_score(targets, preds) * 100.0,
        "Recall (%)": recall_score(targets, preds) * 100.0,
        "Specificity (%)": (cm[0,0]/(cm[0,0]+cm[0,1]))*100.0,
        "F1-Score (%)": f1_score(targets, preds) * 100.0,
        "ROC-AUC": roc_auc_score(targets, probs),
        "Confusion Matrix": cm
    }

# Display Summary Table
df_summary = pd.DataFrame(test_metrics).T
display(df_summary[["Accuracy (%)", "Recall (%)", "Specificity (%)", "F1-Score (%)", "ROC-AUC"]])

## Step 5: Side-by-Side Confusion Matrices & Learning Curves

In [ ]:
# 1. Plot Confusion Matrices
fig, axes = plt.subplots(1, len(trained_models), figsize=(5 * len(trained_models), 4))
for ax, (name, m) in zip(axes, test_metrics.items()):
    sns.heatmap(m["Confusion Matrix"], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 12, "weight": "bold"})
    ax.set_title(f"{name}\nAcc: {m['Accuracy (%)']:.1f}% | Rec: {m['Recall (%)']:.1f}%")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# 2. Plot Learning Curves
plt.figure(figsize=(10, 4))
for name, h in histories.items():
    plt.plot(range(1, EPOCHS + 1), h["val_acc"], label=f"{name} (Val Acc)", lw=2)
plt.title(f"Model Validation Accuracy Across {EPOCHS} Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Step 6: 📸 UPLOAD AN IMAGE FILE TO TEST ALL MODELS

Run this cell and click **'Choose Files'** to upload any Chest X-ray image (`.jpeg`, `.png`, `.jpg`). All 4 models will immediately evaluate it side-by-side!

In [ ]:
print("👇 Click 'Choose Files' to upload a chest X-ray image file:")
uploaded_img = files.upload()

for img_name in uploaded_img.keys():
    raw_image = Image.open(img_name).convert('RGB')
    input_tensor = eval_tf(raw_image).unsqueeze(0).to(device)
    
    plt.figure(figsize=(4, 4))
    plt.imshow(raw_image)
    plt.title(f"Uploaded Patient X-Ray: {img_name}")
    plt.axis("off")
    plt.show()
    
    print(f"\n{'Model Architecture':<30} | {'Diagnosis':<15} | {'Confidence':<12} | {'Digital Twin Severity'}")
    print("-" * 80)
    
    for name, model in trained_models.items():
        model.eval()
        with torch.no_grad():
            if "Attentive" in name:
                logits, sev = model(input_tensor, return_severity=True)
                sev_text = f"{sev.item():.1f}% loss of aeration"
            else:
                logits = model(input_tensor)
                sev_text = "N/A (No Dual Head)"
                
            probs = F.softmax(logits, dim=1)[0]
            pred_idx = torch.argmax(probs).item()
            pred_label = class_names[pred_idx]
            conf = probs[pred_idx].item() * 100.0
            
            print(f"{name:<30} | {pred_label:<15} | {conf:>6.1f}%     | {sev_text}")